# W3 (optional) · Policy iteration for the coating machine

This notebook confirms part 3 of W3 for a Markov decision process of any size. It needs only Python with `numpy`.

1. Set `GROUP` to your group number and run all cells.
2. Compare the policy and cost per shift with your Excel answer.

States: Good (G), Minor wear (Mi), Major wear (Ma), Failed (F). In Mi and Ma you may continue or do preventive maintenance (PM), which restores Good before the next shift. Failed is always repaired to Good.

In [ ]:
import numpy as np

GROUP = 1   # change to your group number

GROUP_COSTS = {1: {'cF': 20000, 'cPM': 4000}, 2: {'cF': 18000, 'cPM': 4000}, 3: {'cF': 18000, 'cPM': 4000}, 4: {'cF': 20000, 'cPM': 8000}, 5: {'cF': 18000, 'cPM': 4000}, 6: {'cF': 16000, 'cPM': 5000}, 7: {'cF': 18000, 'cPM': 7000}, 8: {'cF': 16000, 'cPM': 7000}, 9: {'cF': 20000, 'cPM': 4000}, 10: {'cF': 24000, 'cPM': 6000}}
P_CONTINUE = np.array([[0.80, 0.15, 0.04, 0.01],
                       [0.00, 0.85, 0.12, 0.03],
                       [0.00, 0.00, 0.60, 0.40],
                       [1.00, 0.00, 0.00, 0.00]])   # Failed row: repair to Good
RESET = np.array([1.0, 0.0, 0.0, 0.0])
c_fail, c_pm = GROUP_COSTS[GROUP]['cF'], GROUP_COSTS[GROUP]['cPM']
OPERATING = np.array([0, 500, 3000, c_fail], dtype=float)
STATES = ['G', 'Mi', 'Ma', 'F']
ACTIONS = {0: ['continue'], 1: ['continue', 'PM'], 2: ['continue', 'PM'], 3: ['repair']}
print('Failure cost', c_fail, ' PM cost', c_pm)

In [ ]:
def step(state, action):
    """Transition row and cost of one action in one state."""
    if action == 'PM':
        return RESET, c_pm
    return P_CONTINUE[state], OPERATING[state]


def evaluate(policy):
    """Solve g + h_i = c_i + sum_j p_ij h_j with h_G = 0. Returns g (cost per shift) and h."""
    n = len(STATES)
    A = np.zeros((n, n)); b = np.zeros(n)
    for i in range(n):
        p, c = step(i, policy[i])
        A[i, 0] = 1.0                      # unknown g in column 0 (h_G = 0 is dropped)
        for j in range(1, n):
            A[i, j] = (1.0 if i == j else 0.0) - p[j]
        b[i] = c
    x = np.linalg.solve(A, b)
    return x[0], np.concatenate([[0.0], x[1:]])


def improve(h):
    """Choose, in each state, the action with the lowest c + sum_j p_j h_j."""
    new = []
    for i in range(len(STATES)):
        best = min(ACTIONS[i], key=lambda a: step(i, a)[1] + step(i, a)[0] @ h)
        new.append(best)
    return new


policy = ['continue', 'continue', 'continue', 'repair']   # start: run to failure
for it in range(10):
    g, h = evaluate(policy)
    print(f'Iteration {it}: policy in Mi, Ma = {policy[1]}, {policy[2]};  cost per shift = {g:,.1f} CNY')
    new = improve(h)
    if new == policy:
        print('No improvement possible: this policy is optimal.')
        break
    policy = new

In [ ]:
# Check: all four policies by enumeration
for a1 in ('continue', 'PM'):
    for a2 in ('continue', 'PM'):
        g, _ = evaluate(['continue', a1, a2, 'repair'])
        print(f'Mi: {a1:8s}  Ma: {a2:8s}  cost per shift = {g:,.1f}')

**Question.** Policy iteration needed only a few steps here. Why does this matter for a machine with 50 wear states and 3 actions, where enumeration would have 3^50 policies?